# Módulo IX — IA Generativa na Prática

Código do livro *Ciência de Dados com Python* (Howard Roatti). Explicações no livro; repositório: https://github.com/howardroatti/roatti-ciencia-de-dados-python

## Conversando com uma LLM Localmente
### Caminho 1 -- Hugging Face `transformers`

In [ ]:
# OPCIONAL: requer transformers
from transformers import pipeline

# 'text-generation' carrega um modelo decoder-only (Módulo VIII)
gerador = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",   # modelo aberto, versão 'Instruct' (Módulo VIII)
    device_map="auto",                     # usa GPU se houver; senão, CPU
)

mensagens = [
    {"role": "system", "content": "Você é um tutor de ciência de dados, didático e conciso."},
    {"role": "user", "content": "Explique, em uma frase, o que é overfitting."},
]

saida = gerador(mensagens, max_new_tokens=120)
print(saida[0]["generated_text"][-1]["content"])

In [ ]:
# OPCIONAL: requer transformers e torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

nome = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(nome)
modelo = AutoModelForCausalLM.from_pretrained(nome, device_map="auto", torch_dtype="auto")

# 1) o template de chat formata os papéis no padrão que ESTE modelo espera
entrada = tokenizer.apply_chat_template(mensagens, add_generation_prompt=True, return_tensors="pt")

# 2) o modelo gera tokens de forma autorregressiva (Módulo VIII)
with torch.no_grad():
    gerados = modelo.generate(entrada.to(modelo.device), max_new_tokens=120)

# 3) destokenização: de volta a texto legível
print(tokenizer.decode(gerados[0][entrada.shape[-1]:], skip_special_tokens=True))

### Caminho 2 -- Ollama

In [ ]:
# OPCIONAL: requer ollama
import ollama

resposta = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "system", "content": "Você é um tutor de ciência de dados."},
        {"role": "user", "content": "O que é validação cruzada? Responda em 2 frases."},
    ],
)
print(resposta["message"]["content"])

## Controlando a Geração: Amostragem e Seus Parâmetros

In [ ]:
# OPCIONAL: requer o objeto `gerador` da célula do Caminho 1 (transformers)
saida = gerador(
    mensagens,
    max_new_tokens=200,
    do_sample=True,        # habilita amostragem (senão, é greedy)
    temperature=0.7,       # criatividade moderada
    top_p=0.9,             # nucleus sampling
    repetition_penalty=1.1 # penaliza repetição de tokens (evita loops)
)

## Engenharia de *Prompt* (*Prompt Engineering*)
### *Zero-shot*, *Few-shot* e o Poder dos Exemplos

In [ ]:
prompt_fewshot = """Classifique o sentimento como POSITIVO, NEGATIVO ou NEUTRO.

Texto: "Adorei o produto, chegou antes do prazo!"  ->  POSITIVO
Texto: "Veio quebrado e o suporte não respondeu."  ->  NEGATIVO
Texto: "O pacote chegou na terça-feira."           ->  NEUTRO
Texto: "A entrega atrasou, mas o produto é ótimo." ->  """

### Saída Estruturada (JSON)

In [ ]:
# OPCIONAL: requer ollama
import ollama, json

resposta = ollama.chat(
    model="llama3.2:3b",
    messages=[{"role": "user",
               "content": "Extraia nome e idade: 'Ana tem 30 anos.' Responda em JSON."}],
    format={  # esquema JSON: garante saída válida e parseável
        "type": "object",
        "properties": {"nome": {"type": "string"}, "idade": {"type": "integer"}},
        "required": ["nome", "idade"],
    },
)
dados = json.loads(resposta["message"]["content"])
print(dados["nome"], dados["idade"])   # -> Ana 30

## *Embeddings* e Busca Semântica

In [ ]:
# OPCIONAL: requer sentence-transformers
from sentence_transformers import SentenceTransformer
import numpy as np

modelo_emb = SentenceTransformer("all-MiniLM-L6-v2")  # modelo aberto, leve

frases = ["O gato dorme no sofá.",
          "Um felino descansa no móvel.",   # semanticamente próxima da 1ª
          "A bolsa de valores caiu hoje."]  # tema totalmente distinto
vetores = modelo_emb.encode(frases)         # matriz (3, 384): 3 frases, 384 dimensões

def cosseno(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(round(cosseno(vetores[0], vetores[1]), 3))  # alta similaridade (~0.7+)
print(round(cosseno(vetores[0], vetores[2]), 3))  # baixa similaridade (~0.1)

### Bancos de Dados Vetoriais

In [ ]:
# OPCIONAL: requer chromadb
import chromadb

cliente = chromadb.Client()
colecao = cliente.create_collection("meus_docs")

# indexa documentos (o Chroma gera os embeddings internamente)
colecao.add(
    documents=["Python é uma linguagem de programação.",
               "O Pandas serve para análise de dados.",
               "A fotossíntese ocorre nas plantas."],
    ids=["d1", "d2", "d3"],
)

# busca semântica: encontra os 2 documentos mais relevantes
resultado = colecao.query(query_texts=["Como analiso dados tabulares?"], n_results=2)
print(resultado["documents"])   # -> traz o doc do Pandas em primeiro

## RAG -- *Retrieval-Augmented Generation*

In [ ]:
# OPCIONAL: requer sentence-transformers e ollama
from sentence_transformers import SentenceTransformer
import numpy as np, ollama

# --- Fase 1: indexação ---
base = [
    "A política de reembolso da empresa permite devolução em até 30 dias.",
    "O horário de atendimento é de segunda a sexta, das 9h às 18h.",
    "Produtos em promoção não podem ser trocados, apenas reembolsados.",
]
emb = SentenceTransformer("all-MiniLM-L6-v2")
indice = emb.encode(base)                      # embeddings dos fragmentos

# --- Fase 2: consulta ---
pergunta = "Posso devolver um item comprado há 20 dias?"
q = emb.encode(pergunta)
sims = indice @ q / (np.linalg.norm(indice, axis=1) * np.linalg.norm(q))  # cosseno vetorizado
top = sims.argsort()[::-1][:2]                 # 2 fragmentos mais relevantes
contexto = "\n".join(base[i] for i in top)

prompt = f"""Responda à pergunta usando SOMENTE o contexto abaixo.
Se a resposta não estiver no contexto, diga que não há informação.

Contexto:
{contexto}

Pergunta: {pergunta}"""

resposta = ollama.chat(model="llama3.2:3b",
                       messages=[{"role": "user", "content": prompt}])
print(resposta["message"]["content"])

### RAG Completo e Offline: do *Chunking* à Avaliação

In [ ]:
import sys
sys.path.insert(0, "..")          # permite importar src/ a partir de notebooks/
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from src.dados_sinteticos_m8_m9 import (documentos_ficticios, metadados_documentos,
                                        perguntas_ficticias, perguntas_fora_da_base)

docs = documentos_ficticios()          # {id: texto} -- base FICTÍCIA da "Mercado Aurora"
meta = metadados_documentos()
print(len(docs), "documentos;", sum(len(t.split()) for t in docs.values()), "palavras no total")
print(docs["garantia"])

In [ ]:
def fragmentar(texto, tamanho=40, sobreposicao=10):
    """Janelas de `tamanho` palavras; cada janela repete as últimas `sobreposicao` da anterior."""
    palavras = texto.split()
    passo = tamanho - sobreposicao
    return [" ".join(palavras[i:i + tamanho])
            for i in range(0, max(len(palavras) - sobreposicao, 1), passo)]

def indexar(docs, tamanho=40, sobreposicao=10):
    return [{"doc": doc_id, "texto": trecho}
            for doc_id, texto in docs.items()
            for trecho in fragmentar(texto, tamanho, sobreposicao)]

fragmentos = indexar(docs)
print(len(fragmentos), "fragmentos")
for f in fragmentos[:3]:
    print(f"[{f['doc']}] {f['texto']}\n")

In [ ]:
STOP = ("a o as os um uma uns umas de da do das dos em no na nos nas por para com sem que "
        "e ou se ao aos mais mas ja nao ser sao foi vale pode podem posso como quando "
        "qual quais quanto quantos quantas onde voces vocês ha meu meus minha minhas").split()

class RecuperadorTFIDF:
    """Recuperador lexical: TF-IDF + similaridade do cosseno (interface: .buscar)."""
    def __init__(self, fragmentos, **opcoes):
        self.fragmentos = fragmentos
        self.vetorizador = TfidfVectorizer(strip_accents="unicode", lowercase=True,
                                           stop_words=STOP, **opcoes)
        self.matriz = self.vetorizador.fit_transform([f["texto"] for f in fragmentos])

    def buscar(self, pergunta, k=3):
        q = self.vetorizador.transform([pergunta])
        sims = cosine_similarity(q, self.matriz).ravel()
        melhores = sims.argsort()[::-1][:k]
        return [(float(sims[i]), self.fragmentos[i]) for i in melhores]

recuperador = RecuperadorTFIDF(fragmentos)
for score, f in recuperador.buscar("A garantia cobre queda do aparelho?", k=2):
    print(f"{score:.3f}  [{f['doc']}]  {f['texto'][:70]}...")

In [ ]:
def montar_prompt(pergunta, trechos):
    contexto = "\n".join(f"[{i}] ({f['doc']}) {f['texto']}"
                         for i, (_, f) in enumerate(trechos, 1))
    return (
        "Você é o assistente de atendimento da Mercado Aurora.\n"
        "Responda à pergunta usando SOMENTE o contexto abaixo.\n"
        'Se a resposta não estiver no contexto, responda exatamente: "Não encontrei essa '
        'informação na base."\n'
        "Cite a fonte entre colchetes, como [1].\n\n"
        f"Contexto:\n{contexto}\n\nPergunta: {pergunta}"
    )

pergunta = "Quantas tentativas de entrega a transportadora faz?"
print(montar_prompt(pergunta, recuperador.buscar(pergunta, k=2)))

In [ ]:
def _normalizar(texto):
    import unicodedata
    sem_acento = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in sem_acento if unicodedata.category(c) != "Mn")

def _radicais(texto):
    """Palavras de conteúdo reduzidas aos 5 primeiros caracteres (radical grosseiro)."""
    palavras = re.findall(r"[a-z0-9]{2,}", _normalizar(texto))
    return {p[:5] for p in palavras if p not in STOP}

def gerador_simulado(prompt):
    """'LLM' determinística: devolve a frase do contexto que mais compartilha radicais com a
    pergunta. Mesma assinatura de um gerador real: recebe o prompt, devolve texto."""
    contexto, pergunta = prompt.split("Contexto:\n")[1].split("\n\nPergunta: ")
    candidatas = []
    for m in re.finditer(r"^\[(\d+)\] \((.+?)\) (.*)$", contexto, flags=re.M):
        for frase in re.split(r"(?<=[.!?])\s+", m.group(3)):
            candidatas.append((len(_radicais(pergunta) & _radicais(frase)), frase, m.group(1)))
    melhor = max(candidatas, key=lambda c: c[0]) if candidatas else (0, "", "")
    if melhor[0] == 0:
        return "Não encontrei essa informação na base."
    return f"{melhor[1]} [{melhor[2]}]"

def rag(pergunta, recuperador, gerar, k=2, limiar=0.0):
    trechos = recuperador.buscar(pergunta, k=k)
    if trechos[0][0] < limiar:                     # recuperação fraca: nem chama o gerador
        return "Não encontrei essa informação na base.", trechos
    return gerar(montar_prompt(pergunta, trechos)), trechos

for p in ["Quantas tentativas de entrega a transportadora faz?",
          "A central abre aos sábados?"]:
    resposta, _ = rag(p, recuperador, gerador_simulado)
    print(f"P: {p}\nR: {resposta}\n")

In [ ]:
def avaliar(recuperador, perguntas, ks=(1, 3)):
    """Hit rate@k e MRR no nível do DOCUMENTO (vários fragmentos do mesmo documento contam uma vez)."""
    posicoes = []
    for pergunta, alvo in perguntas:
        ranking = []
        for _, f in recuperador.buscar(pergunta, k=len(recuperador.fragmentos)):
            if f["doc"] not in ranking:
                ranking.append(f["doc"])
        posicoes.append(ranking.index(alvo) + 1)    # posição (1 = primeiro) do doc correto
    pos = np.array(posicoes)
    resultado = {f"hit@{k}": float((pos <= k).mean()) for k in ks}
    resultado["MRR"] = float((1 / pos).mean())
    return resultado, pos

perguntas = perguntas_ficticias()
metricas, pos = avaliar(recuperador, perguntas)
print({k: round(v, 3) for k, v in metricas.items()})
for (p, alvo), r in zip(perguntas, pos):
    if r > 1:
        print(f"posição {r}: {p}  (esperado: {alvo})")

In [ ]:
configs = {
    "TF-IDF palavras":        dict(),
    "TF-IDF palavras 1-2":    dict(ngram_range=(1, 2)),
    "TF-IDF caracteres 3-5":  dict(analyzer="char_wb", ngram_range=(3, 5)),
}
linhas = {}
for nome, opcoes in configs.items():
    if opcoes.get("analyzer") == "char_wb":        # stop_words não se aplica a n-gramas de caracteres
        rec = RecuperadorTFIDF.__new__(RecuperadorTFIDF)
        rec.fragmentos = fragmentos
        rec.vetorizador = TfidfVectorizer(strip_accents="unicode", **opcoes)
        rec.matriz = rec.vetorizador.fit_transform([f["texto"] for f in fragmentos])
    else:
        rec = RecuperadorTFIDF(fragmentos, **opcoes)
    linhas[nome] = avaliar(rec, perguntas)[0]
print(pd.DataFrame(linhas).T.round(3))

In [ ]:
linhas = {}
for tamanho in [10, 20, 40, 80, 160]:
    frags = indexar(docs, tamanho, sobreposicao=tamanho // 4)
    metr = avaliar(RecuperadorTFIDF(frags), perguntas)[0]
    linhas[f"{tamanho} palavras ({len(frags)} frags)"] = metr
print(pd.DataFrame(linhas).T.round(3))

In [ ]:
ruins = indexar(docs, tamanho=8, sobreposicao=0)
rec_ruim = RecuperadorTFIDF(ruins)
pergunta = "Quantos meses de garantia têm os eletrônicos?"
for score, f in rec_ruim.buscar(pergunta, k=2):
    print(f"{score:.3f} [{f['doc']}] {f['texto']}")
resposta, _ = rag(pergunta, rec_ruim, gerador_simulado)
print("Resposta com fragmentos de 8 palavras:", resposta)
resposta, _ = rag(pergunta, recuperador, gerador_simulado)
print("Resposta com fragmentos de 40 palavras:", resposta)

In [ ]:
ok = [recuperador.buscar(p, k=1)[0][0] for p, _ in perguntas]
fora = [recuperador.buscar(p, k=1)[0][0] for p in perguntas_fora_da_base()]
print("melhor score, perguntas COM resposta: min %.3f | mediana %.3f" % (min(ok), np.median(ok)))
print("melhor score, perguntas SEM resposta: max %.3f" % max(fora))

recusa = "Não encontrei essa informação na base."
for limiar in [0.0, 0.12, 0.25]:
    recusadas_ok = sum(rag(p, recuperador, gerador_simulado, limiar=limiar)[0] == recusa
                       for p, _ in perguntas)
    vazaram = sum(rag(p, recuperador, gerador_simulado, limiar=limiar)[0] != recusa
                  for p in perguntas_fora_da_base())
    print(f"limiar {limiar:.2f}: recusa {recusadas_ok}/{len(perguntas)} perguntas respondíveis; "
          f"{vazaram}/{len(perguntas_fora_da_base())} perguntas fora da base ainda recebem resposta")

p = "Vocês têm loja física em Curitiba?"
print(p, "->", rag(p, recuperador, gerador_simulado)[0])

In [ ]:
pergunta = "Em quantos dias posso devolver um produto?"
trechos = recuperador.buscar(pergunta, k=3)
for score, f in trechos:
    print(f"{score:.3f} [{f['doc']}] {f['texto'][:60]}...")

# filtro por metadados: só documentos vigentes entram na busca
vigentes = [f for f in fragmentos if meta[f["doc"]]["vigencia"] == "vigente"]
rec_vigente = RecuperadorTFIDF(vigentes)
resposta, trechos = rag(pergunta, rec_vigente, gerador_simulado)
print("\nResposta com filtro de vigência:", resposta)

In [ ]:
# OPCIONAL: requer sentence-transformers, chromadb e ollama (veja os trechos anteriores do módulo)
import chromadb
import ollama
from sentence_transformers import SentenceTransformer

class RecuperadorDenso:
    """Mesma interface do RecuperadorTFIDF: .buscar(pergunta, k) -> [(score, fragmento)]."""
    def __init__(self, fragmentos):
        self.fragmentos = fragmentos
        self.modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
        self.colecao = chromadb.Client().create_collection("aurora", metadata={"hnsw:space": "cosine"})
        self.colecao.add(ids=[str(i) for i in range(len(fragmentos))],
                         embeddings=self.modelo.encode([f["texto"] for f in fragmentos]).tolist(),
                         documents=[f["texto"] for f in fragmentos])

    def buscar(self, pergunta, k=3):
        r = self.colecao.query(query_embeddings=self.modelo.encode([pergunta]).tolist(), n_results=k)
        return [(1 - d, self.fragmentos[int(i)]) for i, d in zip(r["ids"][0], r["distances"][0])]

def gerador_ollama(prompt):
    r = ollama.chat(model="llama3.2:3b", messages=[{"role": "user", "content": prompt}])
    return r["message"]["content"]

recuperador_real = RecuperadorDenso(fragmentos)
print(avaliar(recuperador_real, perguntas)[0])                  # mesma avaliação, outro recuperador
print(rag("Posso pedir para apagarem meus dados?", recuperador_real, gerador_ollama)[0])